In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib
import geopandas as gpd
import jenkspy
from datetime import datetime, timedelta
from statsmodels.tsa.seasonal import seasonal_decompose
import statsmodels
from scipy.stats import pearsonr
import scipy
import prophet
import warnings
warnings.filterwarnings("ignore")
from matplotlib.font_manager import FontProperties
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from scipy.optimize import curve_fit
import matplotlib.colors as colors
import mapclassify
from libpysal.weights.contiguity import Queen
from libpysal import examples
import geopandas as gpd
from esda.moran import Moran_Local
from splot.esda import lisa_cluster

In [ ]:
font = FontProperties(family='DejaVu Sans', weight='bold')


In [ ]:
pop_mp = pd.read_csv("../../../results/measure_bias/phone_00_05/Active_population_bias_nonspatial.csv", encoding="latin1")
pop_mp = pop_mp.rename(columns= {"Total_active_population": "active_mp"})
pop_mp = pop_mp[['Area_code', 'Area_name', 'Total_Population', 'active_mp']]
pop_fb = pd.read_csv("../../../results/measure_bias/facebook_tts/Active_population_bias_nonspatial.csv", encoding="latin1")
pop_fb = pop_fb[['Area_code', 'Total_active_population']]
df = pop_mp.merge(pop_fb, how='left', left_on='Area_code', right_on='Area_code')
df = df.rename(columns= {"Total_active_population": "active_fb"})
df = df.rename(columns= {"Total_Population": "population"})

df = df.sort_values(by="population").reset_index(drop=True)
# df = df.dropna().reset_index(drop=True)

df['coverage_mp'] = df['active_mp']/df['population']
df['coverage_fb'] = df['active_fb']/df['population']

df['bias_mp'] = 1 - df['coverage_mp']
df['bias_fb'] = 1 - df['coverage_fb']

# df = df[df['coverage_fb'] <= 1].reset_index(drop=True)
# df = df[df['coverage_mp'] <= 1].reset_index(drop=True)

df["Area_code"] = df["Area_code"].astype(int)

In [ ]:
muni = gpd.read_file("../../../data/raw/spatial/municipalities_measure_bias/MunicipiosMX.shp")
muni["CVEGEO"] = muni["CVEGEO"].astype(int)

states = gpd.read_file("../../../data/raw/spatial/gadm/States SHP/gadm41_MEX_1.shp").to_crs(muni.crs)


In [ ]:
df_gdf = muni.merge(df, left_on='CVEGEO', right_on='Area_code', how='left')

In [ ]:

fig, ax = plt.subplots(figsize=(10, 7.5))

df_gdf['bias_mp'] = df_gdf['bias_mp'].fillna(1)

viridis = plt.cm.get_cmap('Blues')
norm = colors.PowerNorm(gamma=1.85,  # < 1 boosts low values, >1 compresses them
                        vmin=df_gdf['bias_mp'].min(),
                        vmax=df_gdf['bias_mp'].max())
# norm = plt.Normalize(min(df_gdf['bias_mp']), max(df_gdf['bias_mp'])) 

df_gdf.plot(column='bias_mp', 
            cmap='Blues', 
            norm=norm,
            lw=0,
            ax=ax)

states.plot(ax=ax, facecolor="None", edgecolor='1', lw=0.3)

for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)

# plt.savefig('../plots/map-bias-mp.pdf', bbox_inches='tight')

    
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))

df_gdf['bias_fb'] = df_gdf['bias_fb'].fillna(1)

viridis = plt.cm.get_cmap('Reds')
norm = colors.PowerNorm(gamma=1.85,  # < 1 boosts low values, >1 compresses them
                        vmin=df_gdf['bias_fb'].min(),
                        vmax=df_gdf['bias_fb'].max())
# norm = plt.Normalize(min(df_gdf['bias_fb']), max(df_gdf['bias_fb'])) 

df_gdf.plot(column='bias_fb', 
            cmap='Reds', 
            norm=norm,
            lw=0,
            ax=ax)

states.plot(ax=ax, facecolor="None", edgecolor='w', lw=0.4)

for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)

# plt.savefig('../plots/map-bias-fb.pdf', bbox_inches='tight')

    
plt.show()

In [ ]:
biasw_mp = pd.read_csv("../../../data/processed/explain_bias/blocked_cv/Active_population_bias_queen_p0005.csv", encoding="latin1")
biasw_fb = pd.read_csv("../../../data/processed/explain_bias/blocked_cv/Active_population_bias_queen_tts.csv", encoding="latin1")


In [ ]:
biasw_mp = biasw_mp[biasw_mp['bias']!=0].reset_index(drop=True)
biasw_mp = biasw_mp[biasw_mp['bias']>0.965].reset_index(drop=True)
biasw_mp = biasw_mp[biasw_mp['bias_w']!=0].reset_index(drop=True)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))

viridis = plt.cm.get_cmap('Blues')
norm = plt.Normalize(0, len(biasw_mp) - 1)   
for i in range(len(biasw_mp)):
    color = viridis(norm(i)**9)
    ax.scatter(biasw_mp.loc[i, 'bias'], biasw_mp.loc[i, 'bias_w'], s=35, alpha=0.6, facecolor="None", edgecolor='darkblue', zorder=i)
    ax.scatter(biasw_mp.loc[i, 'bias'], biasw_mp.loc[i, 'bias_w'], s=35, alpha=0.9, c=color, edgecolor="None", zorder=i)

def lin_func(x, a, b):
    return a + x*b
xdata = biasw_mp["bias"].values
ydata = biasw_mp["bias_w"].values
popt, pcov = curve_fit(lin_func, xdata, ydata, p0=[1, 1])
a_hat, b_hat = popt
fit = lin_func(biasw_mp["bias"], a_hat, b_hat)
ax.plot(biasw_mp["bias"], fit, ls="-", alpha=1, lw=4, c='darkblue', zorder=i+1)
print(a_hat, b_hat)

ax.set_xlim([0.976, 1.001])
ax.set_ylim([0.979, 1.001])

ax.set_yticks([0.980, 0.985, 0.990, 0.995, 1.000])

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(font)

for x in ax.get_xticks():
    ax.axvline(x, color='lightgray', linewidth=0.6, linestyle='-', zorder=1)
for y in ax.get_yticks():
    ax.axhline(y, color='lightgray', linewidth=0.6, linestyle='-', zorder=1)
    
for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis='both', which='both', width=0, length=0, labelsize=20, pad=3)

# plt.savefig('../plots/moran-plot-mp.pdf', bbox_inches='tight')


plt.show()

In [ ]:
biasw_fb = biasw_fb[biasw_fb['bias']>0].reset_index(drop=True)
# biasw_fb = biasw_fb[biasw_fb['bias']>0.965].reset_index(drop=True)
biasw_fb = biasw_fb[biasw_fb['bias_w']>0].reset_index(drop=True)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))

viridis = plt.cm.get_cmap('Reds')
norm = plt.Normalize(0, len(biasw_fb) - 1)   
for i in range(len(biasw_fb)):
    color = viridis(norm(i)**9)
    ax.scatter(biasw_fb.loc[i, 'bias'], biasw_fb.loc[i, 'bias_w'], s=35, alpha=0.6, facecolor="None", edgecolor='crimson', zorder=i)
    ax.scatter(biasw_fb.loc[i, 'bias'], biasw_fb.loc[i, 'bias_w'], s=35, alpha=0.9, c=color, edgecolor="None", zorder=i)

def lin_func(x, a, b):
    return a + x*b
xdata = biasw_fb["bias"].values
ydata = biasw_fb["bias_w"].values
popt, pcov = curve_fit(lin_func, xdata, ydata, p0=[1, 1])
a_hat, b_hat = popt
fit = lin_func(biasw_fb["bias"], a_hat, b_hat)
ax.plot(biasw_fb["bias"], fit, ls="-", alpha=1, lw=4, c='crimson', zorder=i+1)
print(a_hat, b_hat)

ax.set_xlim([-0.04, 1.04])
ax.set_ylim([0.47, 1.03])

ax.set_yticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0], ['0.500', '0.600', '0.700', '0.800', '0.900', '1.000'])

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(font)

for x in ax.get_xticks():
    ax.axvline(x, color='lightgray', linewidth=0.6, linestyle='-', zorder=1)
for y in ax.get_yticks():
    ax.axhline(y, color='lightgray', linewidth=0.6, linestyle='-', zorder=1)
    
for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis='both', which='both', width=0, length=0, labelsize=20, pad=3)

# plt.savefig('../plots/moran-plot-fb.pdf', bbox_inches='tight')


plt.show()

In [ ]:
y = df_gdf['bias_mp'].values
w = Queen.from_dataframe(df_gdf)
w.transform = 'r'
moran_loc = Moran_Local(y, w)

df_gdf['moran_loc_mp'] = moran_loc.Is
df_gdf['q_mp'] = moran_loc.q
df_gdf['p_sim_mp'] = moran_loc.p_sim

In [ ]:

fig, ax = plt.subplots(figsize=(10, 7.5))

color_dict = {
    1: '#e3701e',   
    2: '#9de7eb',   
    3: '#0374a1',   
    4: '#edbe95',   
}

# Map colors to each row based on q
df_gdf['colors_mp'] = df_gdf['q_mp'].map(color_dict)
df_gdf['colors_mp'] = np.where(df_gdf['p_sim_mp'] > 0.05, '#424242', df_gdf['colors_mp'])

df_gdf.plot(df_gdf['colors_mp'], color=df_gdf['colors_mp'], lw=0, ax=ax)

states.plot(ax=ax, facecolor="None", edgecolor='w', lw=0.4)

for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)

plt.savefig('../plots/lisa-mp.pdf', bbox_inches='tight')

    
plt.show()

In [ ]:
y = df_gdf['bias_fb'].values
w = Queen.from_dataframe(df_gdf)
w.transform = 'r'
moran_loc = Moran_Local(y, w)

df_gdf['moran_loc_fb'] = moran_loc.Is
df_gdf['q_fb'] = moran_loc.q
df_gdf['p_sim_fb'] = moran_loc.p_sim

In [ ]:

fig, ax = plt.subplots(figsize=(10, 7.5))

color_dict = {
    1: '#e3701e',   
    2: '#9de7eb',   
    3: '#0374a1',   
    4: '#edbe95',   
}

# Map colors to each row based on q
df_gdf['colors_fb'] = df_gdf['q_fb'].map(color_dict)
df_gdf['colors_fb'] = np.where(df_gdf['p_sim_fb'] > 0.05, '#424242', df_gdf['colors_fb'])

df_gdf.plot(df_gdf['colors_fb'], color=df_gdf['colors_fb'], lw=0, ax=ax)

states.plot(ax=ax, facecolor="None", edgecolor='w', lw=0.4)

for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)

plt.savefig('../plots/lisa-fb.pdf', bbox_inches='tight')

    
plt.show()